In [1]:
from pathlib import Path 
import numpy as np 
import cv2

from scipy import ndimage
import numpy as np 
from PIL import Image

In [2]:
data_dir = Path("../thermal/raw_data").expanduser()
all_files = list(data_dir.glob("*"))
print(len(all_files))

0


In [2]:

data_path = Path("/Volumes/Pluggable_1TB/thermal_images/archive/").expanduser()


thermal_paths = list(data_path.glob("set*/V*/lwir"))


all_files = []

for lwir_dir in thermal_paths:
    if lwir_dir.is_dir():
        files = list(lwir_dir.iterdir()) 
        all_files.extend(files)

In [6]:
img = cv2.imread(str(all_files[0]), cv2.IMREAD_GRAYSCALE)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
edges = clahe.apply(img)
edges = cv2.Canny(edges, 100, 130)
image = edges[:,:,None]
image = np.concatenate([image,image,image], axis=2)
canny_image = Image.fromarray(image)
canny_image.save("outputs/edges.png")

In [1]:
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
)
import numpy as np
import torch

In [4]:
# Control Net model
try:
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-canny", 
        torch_dtype=torch.float16
    )
except Exception as error:
    print (error)

print ('load controlnet')

# --------- LOAD MAIN PIPELINE ----------
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    controlnet=controlnet, 
    dtype=torch.float16
)


pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
/Users/fred/Documents/University of Leeds/Year 3/1/Dist/repo/Thermal_edge_detection/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Unable to load weights from checkpoint file for '/Users/fred/.cache/huggingface/hub/models--lllyasviel--sd-controlnet-canny/snapshots/7f2f69197050967007f6bbd23ab5e52f0384162a/diffusion_pytorch_model.safetensors' at '/Users/fred/.cache/huggingface/hub/models--lllyasviel--sd-controlnet-canny/snapshots/7f2f69197050967007f6bbd23ab5e52f0384162a/diffusion_pytorch_model.safetensors'. 
load controlnet


NameError: name 'controlnet' is not defined

In [ ]:
# --------- INFERENCE ----------
prompt = "high resolution, realistic image, photograph"

out = pipe(
    prompt=prompt,
    negative_prompt="ugly, deformed, blurry, low quality, monochrome",
    image=load_image("outputs/edges.png"),
    num_inference_steps=30,   # reduced for speed
)

# --------- SAVE RESULT ----------
out.images[0].save("reconstructed.png")